In [7]:
from langchain.agents import create_agent
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from langchain_core.tools import tool
from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx
import urllib3
import ssl

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal
import uuid
import base64




# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

# Disable SSL warnings for urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai =  ChatOpenAI()
from langchain_community.utilities.sql_database import SQLDatabase

# Use hardcoded absolute path to the database file
sql_db = SQLDatabase.from_uri("sqlite:///C:/projects/personal/langchn_practise/SalesDB/sales.db")
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=sql_db, llm=llm_openai)
toolkit.get_tools()
from langchain.agents import create_agent

# Disable graph rendering to avoid SSL issues with mermaid.ink
import os
os.environ["LANGCHAIN_DISABLE_GRAPH"] = "true"

agent = create_agent(llm_openai, toolkit.get_tools())
#agent
example_query = "How much total sales we made for Tablet"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How much total sales we made for Tablet
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (chatcmpl-tool-b835f755095e3d00)
 Call ID: chatcmpl-tool-b835f755095e3d00
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

orders
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (chatcmpl-tool-a1a6475d8c97512e)
 Call ID: chatcmpl-tool-a1a6475d8c97512e
  Args:
    table_names: orders, order_items, products
================================= Tool Message =================================
Name: sql_db_schema

Error: table_names {'products', 'order_items'} not found in database
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (chatcmpl-tool-928bad7362c45f95)
 Call ID: ch